In [ ]:
import pandas as pd, numpy as np
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

df = pd.read_csv("../data/raw/transactions-2026-11-04.csv", low_memory=False)
df["INSTANCE_DATE"] = pd.to_datetime(df["INSTANCE_DATE"], errors="coerce")
sales = df[df["GROUP_EN"] == "Sales"].copy()

for col in ["PROCEDURE_EN","IS_OFFPLAN_EN","IS_FREE_HOLD_EN","USAGE_EN",
            "PROP_TYPE_EN","PROP_SB_TYPE_EN","ROOMS_EN","PARKING"]:
    print(f"\n--- {col} (Sales) ---")
    print(sales[col].value_counts(dropna=False).head(8).to_string())

In [ ]:
rep_s = sales[sales["TRANSACTION_NUMBER"].duplicated(keep=False)]
chk = rep_s.groupby("TRANSACTION_NUMBER").agg(
    rows=("TRANS_VALUE", "size"),
    distinct_values=("TRANS_VALUE", "nunique"),
    distinct_areas=("ACTUAL_AREA", "nunique"))

print("Repeated Sales transaction numbers:", len(chk))
print("Identical rows inside Sales:", sales.duplicated().sum())
print("% of repeated numbers with ONE value on all rows:",
      round((chk["distinct_values"] == 1).mean() * 100, 1))
print(chk.head(10))

In [ ]:
big = chk.sort_values("rows", ascending=False)
print("Total Sales rows inside repeated numbers:", chk["rows"].sum())
print(big.head(10))

# Which columns differ between rows of the same repeated Sales transaction?
diff_cols = [c for c in sales.columns if c != "TRANSACTION_NUMBER"]
differs = rep_s.groupby("TRANSACTION_NUMBER")[diff_cols].nunique(dropna=False).gt(1).sum()
print("\nColumns that differ (count of transactions where it differs):")
print(differs[differs > 0].sort_values(ascending=False))

In [ ]:
cols = ["TRANSACTION_NUMBER","INSTANCE_DATE","AREA_EN","PROP_TYPE_EN",
        "PROP_SB_TYPE_EN","TRANS_VALUE","ACTUAL_AREA"]
print("Rows in the Dubai Marina transaction:",
      (df["TRANSACTION_NUMBER"] == "11-31149-2026").sum())
print(df[df["TRANSACTION_NUMBER"] == "11-31149-2026"][cols])

print("\nLowest 10 Sales values:")
print(sales.nsmallest(10, "TRANS_VALUE")[cols])

In [ ]:
SQM_TO_SQFT = 10.7639   # 1 square metre = 10.7639 square feet

units = sales[sales["PROP_TYPE_EN"] == "Unit"].copy()
units["price_per_sqft"] = units["TRANS_VALUE"] / (units["ACTUAL_AREA"] * SQM_TO_SQFT)

print(units["ACTUAL_AREA"].describe(percentiles=[.01, .5, .99]).round(1))
print(units["price_per_sqft"].describe(percentiles=[.01, .05, .5, .95, .99]).round(0))

In [ ]:
key = df["AREA_EN"].str.strip().str.replace(r"\s+", " ", regex=True).str.lower()
variants = df.groupby(key)["AREA_EN"].agg(lambda s: sorted(s.unique()))
print(variants[variants.str.len() > 1].to_string())